In [1]:
import os, re
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd


In [2]:
CLASSES = ["High","Good","Moderate","Bad","Poor"]
C = len(CLASSES); CLS2ID = {c:i for i,c in enumerate(CLASSES)}

def tidy_label(x):
    if x is None or (isinstance(x,float) and pd.isna(x)): return None
    s = str(x).strip().lower().replace('_',' ').replace('-',' ')
    s = s.title()
    return {"High":"High","Good":"Good","Moderate":"Moderate","Bad":"Bad","Poor":"Poor"}.get(s, None)

def parse_acc_from_name(name: str) -> float:
    m = list(re.finditer(r'acc\s*=\s*([0-9]+(?:\.[0-9]+)?)\s*%?', name, flags=re.I))
    if not m: raise ValueError(f"Missing acc=... in: {name}")
    s = m[-1].group(1); v = float(s)
    has_pct = '%' in name[m[-1].start(): m[-1].end()]
    if has_pct or v > 1.0: v /= 100.0
    return float(np.clip(v, 1e-3, 1-1e-3))

def load_votes(pred_dir: Path) -> Tuple[pd.Index, List[str], List[float], List[pd.Series]]:
    paths = sorted([p for p in pred_dir.iterdir() if p.suffix.lower() in (".csv",".tsv",".parquet",".feather")])
    if not paths: raise RuntimeError(f"No files in {pred_dir}")
    file_names, accs, series = [], [], []
    for p in paths:
        if p.suffix.lower()==".csv":      df = pd.read_csv(p)
        elif p.suffix.lower()==".tsv":    df = pd.read_csv(p, sep="\t")
        elif p.suffix.lower()==".parquet":df = pd.read_parquet(p)
        elif p.suffix.lower()==".feather":df = pd.read_feather(p)
        else: continue
        if "SamplingOperations_code" in df.columns:
            df = df.set_index(df["SamplingOperations_code"].astype(str)).drop(columns=["SamplingOperations_code"])
        else:
            if df.index.name != "SamplingOperations_code":
                raise ValueError(f"{p.name} lacks SamplingOperations_code")
            df.index = df.index.astype(str)
        # detect label col
        cand = [c for c in df.columns if c not in ("SamplingOperations_code",)]
        if len(cand)==1: col = cand[0]
        else:
            for name in ["IBD_EQR_Status","predictions","label","class","y_hat","y"]:
                if name in df.columns: col = name; break
            else: raise ValueError(f"Cannot infer label column in {p.name}")
        s = df[col].map(tidy_label)
        series.append(s)
        file_names.append(p.name)
        accs.append(parse_acc_from_name(p.name))
    # union index
    items = pd.Index(sorted(set().union(*[set(s.index) for s in series])), name="SamplingOperations_code")
    return items, file_names, accs, series


# integer programmitng

In [ ]:
import pulp

def solve_consensus_exact(items: pd.Index,
                          file_names: List[str],
                          accs: List[float],
                          series: List[pd.Series],
                          round_mode: str = "nearest",
                          allow_slack: bool = False,
                          time_limit: int | None = None):
    """
    Binary x[i,c] with per-model exact agreement constraints:
      sum_{i in S_k} x[i, label_k(i)] = t_k  where t_k = round(a_k * |S_k|)
    If allow_slack=True, introduces integer slack and minimizes total slack.
    """
    N = len(items); K = len(file_names)
    # map labels to ints per model
    Lk = []
    Sk = []
    nk = []
    tk = []
    for k, s in enumerate(series):
        a = accs[k]
        s_al = s.reindex(items)
        idx = s_al.dropna().index
        Sk.append(idx)
        nk.append(len(idx))
        if round_mode == "nearest":
            tk.append(int(np.rint(a * len(idx))))
        elif round_mode == "floor":
            tk.append(int(np.floor(a * len(idx))))
        elif round_mode == "ceil":
            tk.append(int(np.ceil(a * len(idx))))
        else:
            raise ValueError("round_mode in {'nearest','floor','ceil'}")
        lab_ids = s_al.loc[idx].map(lambda z: CLS2ID[z]).astype(int).values
        Lk.append(pd.Series(lab_ids, index=idx))

    # model
    prob = pulp.LpProblem("ConsensusExact", pulp.LpMinimize if allow_slack else pulp.LpMinimize)

    # variables x[i,c] ∈ {0,1}
    X = {(i,c): pulp.LpVariable(f"x_{i}_{c}", lowBound=0, upBound=1, cat="Binary")
         for i in range(N) for c in range(C)}

    # per-item assignment
    for i in range(N):
        prob += pulp.lpSum(X[(i,c)] for c in range(C)) == 1, f"one_class_{i}"

    # agreement constraints per model
    slack_vars = []
    for k in range(K):
        # build sum over its covered items
        if nk[k] == 0:
            continue
        rows = [items.get_loc(idx) for idx in Sk[k]]
        labs = Lk[k].values  # aligned with rows
        agree_sum = pulp.lpSum(X[(rows[j], labs[j])] for j in range(len(rows)))
        if allow_slack:
            sp = pulp.LpVariable(f"s_plus_{k}", lowBound=0, cat="Integer")
            sm = pulp.LpVariable(f"s_minus_{k}", lowBound=0, cat="Integer")
            slack_vars += [sp, sm]
            prob += agree_sum + sm - sp == tk[k], f"acc_{k}"
        else:
            prob += agree_sum == tk[k], f"acc_{k}"

    # objective
    if allow_slack:
        prob += pulp.lpSum(slack_vars)
    else:
        prob += 0

    # solve
    solver = pulp.PULP_CBC_CMD(msg=True, timeLimit=time_limit) if time_limit else pulp.PULP_CBC_CMD(msg=True)
    status = prob.solve(solver)
    if pulp.LpStatus[status] != "Optimal" and pulp.LpStatus[status] != "Feasible":
        raise RuntimeError(f"Solver status: {pulp.LpStatus[status]}")

    # extract y_hat
    xmat = np.zeros((N, C), dtype=int)
    for i in range(N):
        for c in range(C):
            xmat[i,c] = int(pulp.value(X[(i,c)]) > 0.5)
    y_idx = xmat.argmax(axis=1)
    y_hat = [CLASSES[j] for j in y_idx]

    # check agreements
    check = []
    for k in range(K):
        if nk[k] == 0:
            check.append({"modelo": file_names[k], "accuracy_reportada": accs[k], "coincidencia_con_df": np.nan})
            continue
        ids = Sk[k]
        rows = [items.get_loc(idx) for idx in ids]
        labs = Lk[k].values
        matches = (y_idx[rows] == labs).mean()
        check.append({"modelo": file_names[k], "accuracy_reportada": accs[k], "coincidencia_con_df": matches})
    check_df = pd.DataFrame(check)
    return y_hat, y_idx, xmat, check_df


In [4]:
# set your directory
dir_win   = r"notebooks\05 David-Skene EM\results"
dir_posix = "notebooks/05 David-Skene EM/results"
pred_dir = Path(dir_win) if Path(dir_win).exists() else Path(posix_like:=dir_posix)

items, file_names, accs, series = load_votes(pred_dir)

# exact enforcement; change round_mode if you prefer floor/ceil
y_hat, y_idx, Xbin, check_df = solve_consensus_exact(
    items, file_names, accs, series,
    round_mode="nearest",
    allow_slack=True,      # set True if infeasible, solver will minimize slack
    time_limit=None
)

# Write outputs
out = pd.DataFrame({"y_hat": y_hat}, index=items)
out.to_csv(pred_dir / "Yhat_IntegerProgramming.csv")
check_df.to_csv(pred_dir / "agreement_check.csv", index=False)
display(check_df)
print("Saved:", pred_dir / "Yhat_IntegerProgramming.csv")
print("Saved:", pred_dir / "agreement_check.csv")


,modelo,accuracy_reportada,coincidencia_con_df
0,BC_acc=0.820462.csv,0.820462,0.820413
1,CBCr_acc=0.833695.csv,0.833695,0.833657
2,CBph_acc=0.852459.csv,0.852459,0.852375
3,CBt_acc=0.8689.csv,0.868900,0.868974
4,Interpolation_acc=0.311300.csv,0.311300,0.311319
5,JAPredictionsXGB_acc=0.816907.csv,0.816907,0.816882
6,RandomPredictions_acc=0.3075.csv,0.307500,0.307525
7,RFG_acc=0.814100.csv,0.814100,0.814142
8,RFpR_acc=0.802489.csv,0.802489,0.802402
9,XGBTruncatedClean_acc=0.372300.csv,0.372300,0.372241


Saved: notebooks\05 David-Skene EM\results\Yhat_IntegerProgramming.csv
Saved: notebooks\05 David-Skene EM\results\agreement_check.csv
